# Face Recognition Project

## 1. Introduction

## 2. Get Setup

### 2.1 Download Dataset Helper Function

In [ ]:
import os
import zipfile
from pathlib import Path
import requests

def download_data(source: str, 
                    destination: str,
                    remove_source: bool = True) -> Path:
    """Downloads a zipped dataset from source and unzips to destination.

    Args:
        source (str): A link to a zipped file containing data.
        destination (str): A target directory to unzip data to.
        remove_source (bool): Whether to remove the source after downloading and extracting.
    
    Returns:
        pathlib.Path to downloaded data.
    
    Example usage:
        download_data(source="https://github.com/mrdbourke/pytorch-deep-learning/raw/main/data/pizza_steak_sushi.zip",
        destination="pizza_steak_sushi")
    """
    # Setup path to data folder
    data_path = Path("data/")
    image_path = data_path / destination

    # If the image folder doesn't exist, download it and prepare it... 
    if image_path.is_dir():
        print(f"[INFO] {image_path} directory exists, skipping download.")
    else:
        print(f"[INFO] Did not find {image_path} directory, creating one...")
        image_path.mkdir(parents=True, exist_ok=True)
        
        # Download pizza, steak, sushi data
        target_file = Path(source).name
        print(f"[INFO] Downloading {target_file} from {source}...")
        with requests.get(source, stream=True, timeout=30) as response:
            response.raise_for_status()
            with open(data_path / target_file, "wb") as f:
                for chunk in response.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)

        # Unzip pizza, steak, sushi data
        with zipfile.ZipFile(data_path / target_file, "r") as zip_ref:
            print(f"[INFO] Unzipping {target_file} data...") 
            zip_ref.extractall(image_path)

        # Remove .zip file
        if remove_source:
            os.remove(data_path / target_file)
    
    return image_path

### 2.2 Unzip Dataset

In [ ]:
from pathlib import Path
import zipfile

dataset_dir = Path("dataset")
zip_path = Path("data/dataset.zip")

if dataset_dir.exists():
    print("[INFO] Dataset already available.")
elif zip_path.exists():
    print("[INFO] Unzipping dataset...")
    dataset_dir.mkdir(parents=True, exist_ok=True)
    with zipfile.ZipFile(zip_path, "r") as zip_ref:
        zip_ref.extractall(dataset_dir)
else:
    print("[INFO] Dataset zip not found locally... downloading dataset...")
    download_data(source="https://github.com/Axeloooo/Face-Recognition/raw/devel/data/dataset.zip",
                    destination="dataset")

## 3. Get Data

In [ ]:
import cv2
import numpy as np
from glob import glob
from pathlib import Path

# custom load data function to load images and record subject labels
def get_data(path: str):
    paths = glob(path, recursive=True)
    data = [] #list of images
    label = [] #list of labels

    for path in paths:
        img = cv2.imread(path,0) # read image
        # Extract subject label from path using pathlib (OS-independent)
        path_parts = Path(path).parts
        subject_folder = [part for part in path_parts if part.startswith('s') and part[1:].isdigit()]
        if subject_folder:
            subject_label = subject_folder[0][1:]  # Remove 's' prefix
        else:
            continue  # Skip if subject folder not found

        # pre−processing step
        # can resize, rescale, normalize
        img = img.reshape(-1) # reshape image to a 1D vector
        img = np.float32(img / 255.0) #normalize to 0−1 value

        # can apply LBP, PCA or other forms of feature extraction
        # append images and labels
        data.append(img)

        # decrease all labels by 1 since subject labels start from 1
        label.append(int(subject_label)-1)
        
    return np.array(data), np.array(label)

## 4. Local Binary Pattern (LBP) 

In [ ]:
from skimage.feature import local_binary_pattern
from skimage.io import imread

# Read one image of subject 1 from dataset
img = imread("dataset/s1/1.pgm", as_gray=True)

# Extract LBP feature from the image
# P: Number of circularly symmetric neighbor set points = 12
# Q: Radius of circle = 3
lbp = local_binary_pattern(img, 12, 3)

## 5. Support Vector Machines (SVM)

![](https://github.com/Axeloooo/Face-Recognition/raw/devel/images/support-vector-machines.png)

In [ ]:
from sklearn.svm import SVC
import numpy as np

# TODO: Update these glob patterns if your dataset is stored in a different location.
# Example split for datasets organized like dataset/s1/1.pgm ... dataset/s40/10.pgm
train_path = "dataset/s*/[1-8].pgm"  
test_path = "dataset/s*/[9-10].pgm"

train_data , train_label = get_data(train_path) # use the previous custom get data function
test_data , test_label = get_data(test_path) # use the previous custom get data function

if len(train_data) == 0 or len(train_label) == 0:
    raise ValueError(f"No training data found for pattern: {train_path}. Update train_path to match your dataset files.")
if len(test_data) == 0 or len(test_label) == 0:
    raise ValueError(f"No test data found for pattern: {test_path}. Update test_path to match your dataset files.")

svm = SVC(C=5.0, gamma=0.001, probability=True) #experiment with different C and gamma
svm.fit(train_data , train_label)

# probability matrix NxM where N is number of samples and M is the number of classes
probability_matrix = svm.predict_proba(test_data)

# calculate accuracy
prediction = np.argmax(probability_matrix ,1)
result = prediction == test_label
accuracy = np.sum(result)/len(result)

## 6. Multi-Layer Perceptron (MLP)

![](https://github.com/Axeloooo/Face-Recognition/raw/devel/images/multi-layer-perceptron.png)

In [ ]:
from sklearn.neural_network import MLPClassifier
import numpy as np

# TODO: Update these glob patterns if your dataset is stored in a different location.
# Example split for datasets organized like dataset/s1/1.pgm ... dataset/s40/10.pgm
train_path = "dataset/s*/[1-8].pgm"
test_path = "dataset/s*/[9-10].pgm"

# customize get data function to include pre−processing methods (adding PCA or LBP)
train_data , train_label = get_data(train_path) # use the previous custom get data function
test_data , test_label = get_data(test_path) # use the previous custom get data function

if len(train_data) == 0 or len(train_label) == 0:
    raise ValueError(f"No training data found for pattern: {train_path}. Update train_path to match your dataset files.")
if len(test_data) == 0 or len(test_label) == 0:
    raise ValueError(f"No test data found for pattern: {test_path}. Update test_path to match your dataset files.")

# create MLP with 3 layers of perceptrons
# first layers has 128 neurons then 64 then another 128
# experiment with different layers/neurons
# experiment with different learning rate
mlp = MLPClassifier(hidden_layer_sizes=(128,64,128),
                    learning_rate_init=0.001,
                    random_state=1)
mlp.fit(train_data , train_label)

# probability matrix NxM where N is number of samples and M is the number of classes
probability_matrix = mlp.predict_proba(test_data)

# calculate accuracy
prediction = np.argmax(probability_matrix ,1)
result = prediction == test_label
accuracy = np.sum(result)/len(result)

## 7. ROC (FPR vs. TPR)

## 8. DET (FPR vs. FNR)

## 9. Conclusion